## 0. Import Python Libraries

An essential first step . . .


In [1]:
# crim intervals
import crim_intervals
import crim_intervals.visualizations as viz

from crim_intervals import CorpusBase, ImportedPiece, importScore


# standard library
import glob
import os
import re
import warnings
from copy import deepcopy

# data analysis
import pandas as pd
from itertools import combinations

# visualization
import altair as alt
import plotly.express as px
import plotly.io as pio

# network analysis
import networkx as nx
from community import community_louvain

from pyvis.network import Network


# configure plotting for Quarto
alt.renderers.enable('default')
pio.renderers.default = "plotly_mimetype+notebook"

# suppress warnings
warnings.filterwarnings('ignore')


### 0.1 Helper Functions and Sorting List

- These will help us sort and regularize note names

# 1. Introduction:  All About CRIM Intervals

In this Notebook we will explore our contrapuntal duos with **CRIM Intervals**, a Python library developed especially for the purpose of analyzing 'contrapuntal' music like the duos we have been considering lately. It leverages the power of  Mike Cuthbert's excellent Python library [music21](https://web.mit.edu/music21/) (which takes care of all the headaches of finding notes and intervals), but brings the results into Pandas, and thus allows us to turn scores into tablular data.  Learn more about [CRIM Intervals](https://github.com/HCDigitalScholarship/intervals).

Also consult the CRIM API reference for [CRIM Intervals](https://hcdigitalscholarship.github.io/intervals/api/main_objs/).

### Notes and Durations
- **piece.notes()**, which finds all the notes and rests in a score, with a tabular score-like representation of the pitches, pitch classes, and durations (expressed in music21 "offsets", in which each quarter note corresponds to the value of 1.0). It can also derminte the location of any note as a measure+beat reference with piece.detailIndex()
- **piece.durations()**, with quarter-note = 1.0, as per music21.  

Learn more about [notes](https://github.com/HCDigitalScholarship/intervals/blob/main/tutorial/02_Notes_Rests.md) and about [durations](https://github.com/HCDigitalScholarship/intervals/blob/main/tutorial/03_Durations.md)

### Melodic and Harmonic Intervals
- **piece.melodic()**, which finds melodic intervals in any voice part, with various options for diatonic, chromatic and zero-based distances. Intervals can be compound (distinguishing between tenths and thirds, for instance), or simple, and can include quality (distinguishing major and minor thirds, for instance), or not.
- **piece.harmonic()**, which finds harmonic intervals between every combination of two voices in a piece, with various options for diatonic and chromatic. These intervals can also be directed (as when a tenor voice sounds above the altus), or not.

Learn more [melodic intervals](https://github.com/HCDigitalScholarship/intervals/blob/main/tutorial/06_Melodic_Intervals.md) and [harmonic intervals](https://github.com/HCDigitalScholarship/intervals/blob/main/tutorial/07_Harmonic_Intervals.md)


### Ngrams
- **piece.ngrams()**, which finds n-grams of any length in each voice part. n-grams are frequently used in linguistic analysis (), and can help us find repeating patterns within and among works.
- The ngram tool can be used for any of the methods above: **melodic**, **harmonic**, **durations**, **lyrics**.
- By default it finds **contrapuntal modules**, which represent in numerical values a combination of the vertical intervals made between any two voices with the melodic intervals heard in the motion of the lower voice. A module of `7_Held 6_-2, 8` for instance, represents **vertical intervals of `7, 6, 8` between two voices** and in the **lower voice a tied note followed by a descending second**. Together these five events represent a typical cadence formula. Repeating modules are a key part of contrapuntal style.

Learn more about [ngrams](https://github.com/HCDigitalScholarship/intervals/blob/main/tutorial/08_Contrapuntal_Modules.md)

### Heatmaps, Graphs, and Networks

CRIM Intervals also features tools that help us visualize musical events in different ways, including:

- **bar charts, histograms, and radar (spider) plots** of events (such as notes, or intervals), which are useful for understand ranges, distributions, and other large-scale distribution of categorical data
- **heatmaps**, showing where in a piece patterns (such as repeating n-grams) occur
- **network diagrams**, showing related patterns or pieces

### Import a single piece

Import a single piece:`




`piece = importScore('/MEI/Bach_BWV_0772.mei')`

Then call any of the methods above with that piece:

`notes = piece.notes()`

This will return a Pandas dataframe of the *notes and rests* in the piece.

### Import a "Corpus" of pieces

Or a set of pieces, which we call a 'corpus':

```python
file_list = ('MEI/Bach_BWV_0772_rev.mei',
             'MEI/Bartok_Mikrokosmos_022_rev.mei',
             'MEI/Morley_1595_01_Go_ye_my_canzonettes_rev.mei')
# now import those files as a Corpus
corpus = CorpusBase(file_list)
```

Then we use the `batch` method to create a list of dfs--one for each piece in the `corpus`. In this case we define a function to use (`func`) then pass in a dictionary keyword arguments (`kwargs`).  The `metadata` parameter allows us to include composer and title information each dataframe of results.  And then we can combine the dfs into a single set.

```python
func = ImportedPiece.melodic  # <- NB there are no parentheses here
list_of_dfs = corpus.batch(func = func, 
                            kwargs = {'kind': 'd', 
                                     'compound' : True}, 
                            metadata=True)
mel = pd.concat(list_of_dfs)

```

### Corpus Tools Make Things Simple!

Note that we also have some special tools that make quick work of applying various methods to a corpus of pieces with corpus tools.  

See [CRIM Intervals Tutorials](https://github.com/HCDigitalScholarship/intervals/blob/main/tutorial/21_Corpus_Tools.md) for more information on these.



# 2.0 Experiment with One File

You can start with just one file so you understand what is happening as CRIM Intervals assembles dataframes.

Load one piece:

```python
# pick a file
file_name = 'MEI/Bach_BWV_0772.mei'

# load a piece object
piece = importScore(file_name)

# run a method and show results

notes = piece.notes()
notes
```

## 2.1 Get Notes

We get the notes with `piece.notes()`, which returns a Pandas dataframe of the notes and rests in the piece.  Here are the options:



```python
notes(combineRests=True, combineUnisons=False)
```

And to make our outputs more consistent we can use `piece.numberParts()` to rename the parts in a consistent way, so that the first part is always called "1", the second "1", etc.  This is especially useful when we are working with a corpus of pieces, where the parts may be named differently in each piece.

```python
notes = piece.notes()
notes = piece.numberParts(notes) 



In [9]:
# pick a file
file_name = 'MEI/Bach_BWV_0772.mei'

# load a piece object
piece = importScore(file_name)

# run a method and show results
notes = piece.notes()
notes = piece.numberParts(notes)  # add part numbers to the notes table

notes

,1,2
0.00,Rest,Rest
0.25,C4,NaN
0.50,D4,NaN
0.75,E4,NaN
1.00,F4,NaN
...,...,...
83.00,D4,G3
83.25,C5,NaN
83.50,F4,G2
83.75,B4,NaN


In [10]:
# We could also get the durations

durs = piece.durations()
durs = piece.numberParts(durs)  # add part numbers to the durations table

# as strings
durs = durs.map(str)

In [11]:
#  add metadata

notes['Composer'] = piece.metadata['composer']
notes['Title'] = piece.metadata['title']


# fill na for clarity
notes.fillna('').head(10)

,1,2,Composer,Title
0.00,Rest,Rest,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
0.25,C4,,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
0.50,D4,,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
0.75,E4,,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
1.00,F4,,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
1.25,D4,,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
1.50,E4,,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
1.75,C4,,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
2.00,G4,,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
2.25,,C3,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"


### Where are the measure numbers?

By default CRIM Intervals reports the locations by "Offset", in which each 1.0 represents 1 quarter note.  These are zero based, so that the first beat is 0.00.

In order to also get the Measure and Beat information, we need to pass the results to a function called `detailIndex()`.  We can also add a  "Progress" column:  a 'float' that represents the place of this event as a 'proportion' of the whole.   0 is the start.  1 is the end of the piece

Here are all the features we can report with the `detailIndex()` function, and how to call it::

```python
piece.detailIndex(df, measure=True, beat=True, offset=False, t_sig=False, key_sig=False, sounding=False, progress=False, lowest=False, highest=False, _all=False)
```

Note that can also call the `detailIndex()` function with `_all=True` to get all the features at once.  This is a good way to explore what is available.


In [12]:

notes_detailed = piece.detailIndex(notes, offset=True, progress=True)

# the measures and beats are recorded in a multi index.  
# Reset it to make these part of the df
notes_detailed_reset = notes_detailed.reset_index()
notes_detailed_reset

,Measure,Beat,Offset,Progress,1,2,Composer,Title
0,1.0,1.00,0.00,0.000000,Rest,Rest,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
1,1.0,1.25,0.25,0.002976,C4,NaN,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
2,1.0,1.50,0.50,0.005952,D4,NaN,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
3,1.0,1.75,0.75,0.008929,E4,NaN,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
4,1.0,2.00,1.00,0.011905,F4,NaN,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
...,...,...,...,...,...,...,...,...
331,21.0,4.00,83.00,0.988095,D4,G3,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
332,21.0,4.25,83.25,0.991071,C5,NaN,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
333,21.0,4.50,83.50,0.994048,F4,G2,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
334,21.0,4.75,83.75,0.997024,B4,NaN,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"


## 2.2  Visualize the Results

In [13]:
# let's melt the data around the voice parts to make things easier!
selected_df = notes_detailed_reset
# for melting the data, we will set up value and id variables
value_vars= ['1', '2']
id_vars = [col for col in selected_df.columns if col not in value_vars]

# now melt
selected_df_melted = pd.melt(selected_df, value_vars=value_vars, id_vars=id_vars)
selected_df_melted.rename(columns={'variable': 'Voice', 'value': 'Note'}, inplace=True)
selected_df_melted

# # Option to remove rests
df_clean = selected_df_melted[selected_df_melted['Note'] != 'Rest']


# Count notes by Composer Title Note and Voice
note_counts = selected_df_melted.groupby(['Composer', 'Title', 'Note', 'Voice']).size().reset_index(name='Count')
note_counts.head()

,Composer,Title,Note,Voice,Count
0,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",A2,2,3
1,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",A3,2,21
2,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",A4,1,23
3,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",A4,2,5
4,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",A5,1,10


## 2.3 Visualize as Bar Chart



In [14]:
from crim_intervals.sorting_lists import sort_pitch_values


 # Create the bar chart
df = note_counts

# sort the notes that actually occur in this piece, low to high
note_order = sort_pitch_values(df['Note'].unique().tolist())

fig = px.bar(
    df,
    x='Note',
    y='Count',
    color='Voice',
    title=f"Note Distribution Analysis - {piece.metadata['composer']}:  {piece.metadata['title']} ",
    width=800,
    height=600,
    category_orders={'Note': note_order}
)

fig.show()

## 2.4 Visualize as Radar Plot

In [15]:
# get notes
notes = piece.notes()
notes_numbered = piece.numberParts(notes)
notes_numbered['Composer'] = piece.metadata['composer']
notes_numbered['Title'] = piece.metadata['title']


# melt (we don't need measures and offsets!)
selected_df = notes_numbered
# for melting the data, we will set up value and id variables
value_vars= ['1', '2']
id_vars = [col for col in selected_df.columns if col not in value_vars]

# now melt
selected_df_melted = pd.melt(selected_df, value_vars=value_vars, id_vars=id_vars)
selected_df_melted.rename(columns={'variable': 'Voice', 'value': 'Note'}, inplace=True)
selected_df_melted

# # Option to remove rests
df_clean = selected_df_melted[selected_df_melted['Note'] != 'Rest']


# Count notes by Composer Title Note (but NOT voice)
note_counts = selected_df_melted.groupby(['Composer', 'Title', 'Note', ]).size().reset_index(name='Count')

# remove octave designation
note_counts['Note'] = note_counts['Note'].str.replace(r'\d+', '', regex=True)

# Function to standardize note names
def standardize_note(note):
   
    if '-' in note:
        return note.replace('-', 'b')
    return note

note_counts['Note'] = note_counts['Note'].apply(standardize_note)

# now let's scale the counts relative to the total number of notes in the piece
total_notes = note_counts['Count'].sum()
note_counts['Scaled_Count'] = note_counts['Count']/total_notes
note_counts.head()

,Composer,Title,Note,Count,Scaled_Count
0,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",A,3,0.006438
1,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",A,21,0.045064
2,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",A,28,0.060086
3,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",A,10,0.021459
4,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",Bb,4,0.008584


In [16]:
# radar pitch order:  'fifths' or 'chromatic'
fifths = [
    'C', 'B#', 'G', 'D','A', 'E', 'Fb', 'B', 'Cb', 'F#', 'Gb', 'C#', 'Db', 
    'G#', 'Ab', 'D#', 'Eb', 'A#', 'Bb', 'F', 'E#','F##' 
]

In [17]:
df = note_counts.copy()
counted_notes_sorted = df.groupby('Note')['Scaled_Count'].sum().reset_index()

# sort to match pitch_class_order so the line is drawn in the correct sequence
counted_notes_sorted['Note'] = pd.Categorical(counted_notes_sorted['Note'], 
                                               categories=fifths, 
                                               ordered=True)
# drop anything not in the pitch list (e.g. 'Rest'), which would become NaN and break the closed line
counted_notes_sorted = counted_notes_sorted.dropna(subset=['Note'])
# rescale so the remaining pitches sum to 1
counted_notes_sorted['Scaled_Count'] = counted_notes_sorted['Scaled_Count'] / counted_notes_sorted['Scaled_Count'].sum()
counted_notes_sorted = counted_notes_sorted.sort_values('Note').reset_index(drop=True)

# keep only the pitches that occur in this piece, in the order given by fifths
notes_present = [n for n in fifths if n in counted_notes_sorted['Note'].values]


fig = px.line_polar(
    counted_notes_sorted, 
    r='Scaled_Count',                           
    theta='Note',                              
    line_close=True,
    color='Title' if 'Title' in counted_notes_sorted.columns else None,
    category_orders={'Note': notes_present}
)
fig.update_traces(fill='toself', fillcolor='rgba(135, 206, 250, 0.5)')
fig.update_layout(title=f"Note Distribution Analysis - {piece.metadata['composer']}:  {piece.metadata['title']}")
fig.show()

## 2.4 Try the same with Melodic and Harmonic Intervals


### Melodic


With piece.melodic() we can get the melodic intervals in a piece.  Here are the options:

```python
melodic(kind='q', directed=True, compound=True, unit=0, end=True, df=None)
```


This will return all the melodic 'intervals' in the piece. You can select the type of interval with the `kind` parameter:  

- 'd' = diatonic (in which major and minor thirds will simply be '3') 
- 'c' = chromatic (in which we count every half step between successive notes, in this case a C-G will be '7'
- 'q' = with quality (in which major thirds will be M3 and minor thirds will be m3, for example)

Learn more at via the [tutorial](https://github.com/HCDigitalScholarship/intervals/blob/main/tutorial/06_Melodic_Intervals.md)

### Harmonic
Here are the options for harmonic intervals:

```python
harmonic(kind='q', directed=True, compound=True, againstLow=False, df=None)
```


This will return all the melodic 'intervals' in the piece. You can select the type of interval with the `kind` parameter:  

- 'd' = diatonic (in which major and minor thirds will simply be '3') 
- 'c' = chromatic (in which we count every half step between successive notes, in this case a C-G will be '7'
- 'q' = with quality (in which major thirds will be M3 and minor thirds will be m3, for example)

Learn more at via the [tutorial](https://github.com/HCDigitalScholarship/intervals/blob/main/tutorial/07_Harmonic_Intervals.md)
  

You will probably also want to renumber the parts, add measure numbers and composer/title metadata, as shown below!



In [18]:
#  melodic intervals

# here we decide on 'how' the intervals will be reported
mel = piece.melodic(kind = 'd', directed=True, compound=True)

# numbering the parts to make things simpler
mel_numbered = piece.numberParts(mel)

# and the detailed index to show measure, beat, progress
mel_numbered_measures = piece.detailIndex(mel_numbered, offset=False, progress=True).reset_index()

# add the metadata about composer and title
mel_numbered_measures['Composer'] = piece.metadata['composer']
mel_numbered_measures['Title'] = piece.metadata['title']
mel_numbered_measures

,Measure,Beat,Progress,1,2,Composer,Title
0,1.0,1.00,0.000000,Rest,Rest,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
1,1.0,1.50,0.005952,2,NaN,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
2,1.0,1.75,0.008929,2,NaN,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
3,1.0,2.00,0.011905,2,NaN,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
4,1.0,2.25,0.014881,-3,NaN,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
...,...,...,...,...,...,...,...
322,21.0,4.00,0.988095,-2,2,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
323,21.0,4.25,0.991071,7,NaN,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
324,21.0,4.50,0.994048,-5,-8,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
325,21.0,4.75,0.997024,4,NaN,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"


## 2.5 Bar Chart of Melodic Intervals

In [19]:
# for melting the data, we will set up value and id variables
value_vars= ['1', '2']
id_vars = [col for col in mel_numbered_measures.columns if col not in value_vars]

# now melt
mel_numbered_measures_melted = pd.melt(mel_numbered_measures, value_vars=value_vars, id_vars=id_vars)
mel_numbered_measures_melted.rename(columns={'variable': 'Voice', 'value': 'Interval'}, inplace=True)
mel_numbered_measures_melted


# Count notes by Composer Title Note and Voice
mel_counts = mel_numbered_measures_melted.groupby(['Composer', 'Title', 'Interval', 'Voice']).size().reset_index(name='Count')
mel_counts.head()


,Composer,Title,Interval,Voice,Count
0,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",-2,1,81
1,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",-2,2,66
2,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",-3,1,28
3,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",-3,2,22
4,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",-4,1,2


In [20]:
# Create the bar chart
df = mel_counts.copy()

# remove rests
df_no_rests = df[df['Interval'] !=  'Rest']

unique_intervals = df_no_rests['Interval'].unique()

# unique_intervals_no_rest = [interval for interval in unique_intervals if interval != "Rest"]

# sorting function
def sort_integer_strings_explicit(string_array):
    """
    Explicit method: separate negatives and positives, then sort each group
    """
    negatives = [s for s in string_array if int(s) < 0]
    positives = [s for s in string_array if int(s) >= 0]
    
    # Sort negatives in descending order (highest negative first: -1, -2, -3...)
    negatives.sort(key=lambda x: int(x), reverse=False)
    
    # Sort positives in ascending order (lowest positive first: 0, 1, 2, 3...)
    positives.sort(key=lambda x: int(x))
    
    return negatives + positives

interval_order = sort_integer_strings_explicit(unique_intervals)

fig = px.bar(
    df_no_rests,
    x='Interval',
    y='Count',
    color='Voice',
    title=f"Interval Distribution Analysis - {piece.metadata['composer']}:  {piece.metadata['title']} ",
    width=800,
    height=600,

    # note that the pitch order is set earlier in our NB!
    category_orders={'Interval': interval_order}
)

fig.show()

In [14]:
# you could continue to experiment with piece.harmonic, or piece.ngrams

In [21]:
mel = piece.melodic(kind='d')
har = piece.harmonic(kind='d', directed=False)


ng = piece.ngrams(df=har, other=mel, n=4)
det = piece.detailIndex(df=ng).reset_index()
det.head(10)

,Measure,Beat,Part-2_Piano
0,1.0,3.25,"12_2, 14_2, 13_2, 11"
1,1.0,3.50,"14_2, 13_2, 11_-3, 13"
2,1.0,3.75,"13_2, 11_-3, 13_2, 13"
3,1.0,4.00,"11_-3, 13_2, 13_-3, 15"
4,1.0,4.25,"13_2, 13_-3, 15_5, 12"
5,1.0,4.50,"13_-3, 15_5, 12_Held, 8"
6,1.0,4.75,"15_5, 12_Held, 8_-8, 16"
7,2.0,1.00,"12_Held, 8_-8, 16_Held, 17"
8,2.0,3.25,"12_2, 14_2, 13_2, 11"
9,2.0,3.50,"14_2, 13_2, 11_-3, 13"


## 2.6 Heat Map of Ngrams


The ngram method is a powerful tool for finding repeating patterns in music.  It can be used with melodic, harmonic, and duration intervals, as well as with lyrics.  It can also be used to find contrapuntal modules, which are a combination of vertical and melodic intervals.


- Learn about ngrams via the [tutorial](https://github.com/HCDigitalScholarship/intervals/blob/main/tutorial/09_Ngrams_Heat_Maps.md)!

The arguments for the ngram method are as follows:

```python
ngrams(df=None, n=3, how='columnwise', other=None, held='Held', exclude=['Rest'], interval_settings=('d', True, True), unit=0, offsets='first', show_both=False)
```


Here we use ngrams to create a visualization 'heatmap' of the repeating patterns in a piece.  The heatmap shows where in the piece the patterns occur.  Matching patterns have matching colors.

The visualization function takes in various arguments, which in turn are passsed to the ngrams method


- combine_unisons_choice=False:  whether to combine unisons
- kind_choice='d' :  d for diatonic, etc
- directed=True:  directed intervals or not
- compound=True:  compound intervals or not
- length_choice=4:  length of ngram
- include_count=True:  include chart of count of ngrams
- entries_only=True:  limit the ngrams to 'entries' (events after rests or breaks)

In [22]:
# function for ngram heatmap:  do not edit!

def ngram_heatmap(piece, 
                  combine_unisons_choice=False, 
                  kind_choice='d', 
                  directed=True, 
                  compound=True, 
                  length_choice=6, 
                  include_count=False, 
                  entries_only=True):
    # find ngrams
    nr = piece.notes(combineUnisons = combine_unisons_choice)
    mel = piece.melodic(df = nr, 
                        kind = kind_choice,
                        directed = directed,
                        compound = compound,
                        end = False)
    
    # this is for entries only
    if entries_only == True:    
        # pass the following ngrams to the plot below as first df
        entry_ngrams = piece.entries(df = mel, 
                                    n = length_choice, 
                                    thematic = True, 
                                    anywhere = True,
                                    exclude = ['Rest'])
        # pass the ngram durations below to the plot as second df
        entry_ngrams_duration = piece.durations(df = mel, 
                                            n =length_choice, 
                                            mask_df = entry_ngrams)
        # make the heatmap
        chart = viz.plot_ngrams_heatmap(entry_ngrams, 
                                         entry_ngrams_duration, 
                                         selected_patterns=[], 
                                         voices=[],  
                                         includeCount=include_count)
        
        return chart
    
    # this is for ALL mel ngrams (if entries is False in form)
    else:
        mel_ngrams = piece.ngrams(df = mel, n = length_choice, exclude = ['Rest'])
        mel_ngrams_duration = piece.durations(df = mel, 
                                          n =length_choice)
        
        chart = viz.plot_ngrams_heatmap(mel_ngrams, 
                                         mel_ngrams_duration, 
                                         selected_patterns=[], 
                                         voices=[], 
                                         includeCount=include_count)
        
        # # mel_ngrams_detail = piece.detailIndex(mel_ngrams, offset = False)  
        return chart

In [23]:

# duration of the ngrams (so we can see how long they last for our chart!)
nr = piece.notes()
mel = piece.melodic(df = nr, 
                        end = False)
mel_ngrams = piece.ngrams(df = mel, n=4)
mel_ngrams_duration = piece.durations(df = mel, n=4)

mel_ngrams_duration

,Piano,Part-2
0.00,1.0,3.00
0.25,1.0,NaN
0.50,1.0,NaN
0.75,1.0,NaN
1.00,1.0,NaN
...,...,...
82.00,1.0,1.00
82.25,1.0,1.25
82.50,1.0,5.50
82.75,1.0,NaN


### And at last we can visualize the results as a heatmap:

Just pass the `piece` object to the `ngram_heatmap()` function, along with any custom arguments you want to adjust:


```python
ngram_heatmap(piece, 
                  combine_unisons_choice=False, 
                  kind_choice='d', 
                  directed=True, 
                  compound=True, 
                  length_choice=6, 
                  include_count=False, 
                  entries_only=True)
```


In [25]:
ngram_heatmap(piece, 
                  combine_unisons_choice=False, 
                  kind_choice='d', 
                  directed=True, 
                  compound=True, 
                  length_choice=6, 
                  include_count=False, 
                  entries_only=True)

alt.Chart(...)

## 2.7  Contrapuntal Ngrams

- These are ngrams that combine both harmonic and melodic motion to make a 'contrapuntal' pattern
- These are perfect for finding things like cadences (which follow very specific formulas), or repeating complexes of voices
  

In [26]:
# melodic and harmonic dfs
mel = piece.melodic(kind='d')
har = piece.harmonic(kind='d', directed=False)

# number the parts
mel = piece.numberParts(mel)
har = piece.numberParts(har)

# now the ngrams by combining these
ng = piece.ngrams(df=har, other=mel, n=4)

# add details of measure, beats, and progress

ng_det = piece.detailIndex(ng, progress=True).reset_index()

# add metadata
ng_det['composer'] = piece.metadata['composer']
ng_det['title'] = piece.metadata['title']
ng_det.head(10)

,Measure,Beat,Progress,2_1,composer,title
0,1.0,3.25,0.026786,"12_2, 14_2, 13_2, 11","Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
1,1.0,3.50,0.029762,"14_2, 13_2, 11_-3, 13","Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
2,1.0,3.75,0.032738,"13_2, 11_-3, 13_2, 13","Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
3,1.0,4.00,0.035714,"11_-3, 13_2, 13_-3, 15","Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
4,1.0,4.25,0.038690,"13_2, 13_-3, 15_5, 12","Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
5,1.0,4.50,0.041667,"13_-3, 15_5, 12_Held, 8","Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
6,1.0,4.75,0.044643,"15_5, 12_Held, 8_-8, 16","Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
7,2.0,1.00,0.047619,"12_Held, 8_-8, 16_Held, 17","Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
8,2.0,3.25,0.074405,"12_2, 14_2, 13_2, 11","Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"
9,2.0,3.50,0.077381,"14_2, 13_2, 11_-3, 13","Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772"


## Renaissance Cadences

NOTE:  this will result in an error with Bach or Bartok pieces!

In [29]:
# for pieces in Renaissance style, we can use these to find cadences
piece = importScore('MEI/Morley_1595_01_Go_ye_my_canzonettes.mei')
cadences = piece.cadences()
cadences


,CadType,LeadingTones,CVFs,Low,RelLow,Tone,RelTone,TSig,Measure,Beat,Sounding,Progress,SinceLast,ToNext
Last,,,,,,,,,,,,,,
28.0,Evaded Authentic,1,Cb,A4,M3,C,-P4,4/4,8.0,1.0,2,0.129630,28.0,8.0
36.0,Evaded Clausula Vera,1,tC,C5,P5,C,-P4,4/4,10.0,1.0,2,0.166667,8.0,14.0
50.0,Clausula Vera,1,TC,F4,P1,F,-P1,4/4,13.0,3.0,2,0.231481,14.0,16.0
66.0,Evaded Clausula Vera,1,tC,C5,P5,C,-P4,4/4,17.0,3.0,2,0.305556,16.0,14.0
80.0,Evaded Clausula Vera,1,Ct,C5,P5,C,-P4,4/4,21.0,1.0,2,0.370370,14.0,16.0
96.0,Clausula Vera,1,CT,F4,P1,F,-P1,4/4,25.0,1.0,2,0.444444,16.0,60.0
156.0,Clausula Vera,1,CT,F4,P1,F,-P1,4/4,40.0,1.0,2,0.722222,60.0,60.0
216.0,Evaded Clausula Vera,1,Ct,F4,P1,F,-P1,4/4,55.0,1.0,2,1.000000,60.0,4.0


# 3.0 Build a Corpus

Here we obtain the MEI files from our work.

Note that you can use wildcards to limit the corpus to certain files.

```python
all_bach = glob.glob('MEI/Bach*')
all_bach = sorted(all_bach)
```

             
Now Load The List as a CRIM Intervals Corpus           

`corpus = CorpusBase(all_bach)`

In [30]:
# for example
all_bach = glob.glob('MEI/Bach*')
all_bach = sorted(all_bach)
corpus = CorpusBase(all_bach)

## 3.1 Applying CRIM Intervals Functions to Each Piece in a Corpus

Here we apply the _same_ function to each piece in a corpus and then return a combined df that represents our results.  With this we can filter, group, as well as make charts and graphs to show _shared_ patterns across our corpus.

To learn more about how this works, see the [Corpus Tutorial](https://github.com/HCDigitalScholarship/intervals/blob/main/tutorial/01_Introduction_and_Corpus.md#batch-methods-for-a-corpus-of-pieces) from the CRIM Intervals documentation.

Fortunately for you, the `CorpusBase` object now has its own built-in methods that make easy work of all of this.  Each one runs the corresponding piece method on every score in the corpus and returns a single combined df, with `Composer`, `Title` and `Date` columns so you can tell the pieces apart.  See the full list (with all parameters) in the [CRIM Intervals API](https://hcdigitalscholarship.github.io/intervals/api/main_objs/).

All you need to do:

```python
# define your corpus
all_bach = glob.glob('MEI/Bach*')
all_bach = sorted(all_bach)
corpus = CorpusBase(all_bach)

# call the method on the corpus; the arguments determine the settings
corpus.notes(combine_unisons_choice=True, combine_rests_choice=False)
```

Here are the main corpus methods, in brief:

- `corpus.notes()`: table of notes and rests, one row per attack
- `corpus.note_scaled()`: note counts, plus scaled (proportional) counts
- `corpus.note_durs()`: note durations
- `corpus.note_weights()`: pitch-class weights by duration
- `corpus.mel()`: melodic intervals
- `corpus.har()`: harmonic intervals
- `corpus.contrapuntal_ngrams()`: contrapuntal n-grams
- `corpus.melodic_ngrams()`: melodic n-grams
- `corpus.melodic_durational_ratios_ngrams()`: melodic n-grams with durational ratios
- `corpus.harmonic_ngrams()`: harmonic n-grams
- `corpus.sonority_ngrams()`: sonority n-grams (plus bassline)
- `corpus.cadences()`: cadences
- `corpus.presentation_types()`: presentation types

Most of these take the same settings as their single-piece versions (for example `kind_choice`, `compound_choice`, `ngram_length`).  Many also accept `key_sig=True` (to add each piece's key signature) and `include_final=True` (to add each piece's final).

For any other piece method, use `corpus.batch()`.  It returns a _list_ of dfs (one per piece), which you can combine with `pd.concat`:

```python
list_of_dfs = corpus.batch(ImportedPiece.notes, kwargs={'combineUnisons': True})
combined_df = pd.concat(list_of_dfs)
```

## 3.2  Corpus Notes

In [31]:
# define your corpus

selected_pieces = ['MEI/Bach_BWV_0772.mei', 'MEI/Bach_BWV_0774.mei', 'MEI/Bach_BWV_0781.mei']

corpus = CorpusBase(selected_pieces)

# now we can get the notes for the corpus, and combine unisons but not rests
corpus_note_df = corpus.notes(combine_unisons_choice=True, combine_rests_choice=False)
corpus_note_df

,Composer,Title,Date,1,2
0.00,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",None,Rest,Rest
0.25,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",None,C4,NaN
0.50,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",None,D4,NaN
0.75,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",None,E4,NaN
1.00,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",None,F4,NaN
...,...,...,...,...,...
140.50,"Bach, Johann Sebastian","Invention No. 10 in G major, BWV 781",None,B3,NaN
141.00,"Bach, Johann Sebastian","Invention No. 10 in G major, BWV 781",None,G3,NaN
141.50,"Bach, Johann Sebastian","Invention No. 10 in G major, BWV 781",None,NaN,B2
142.00,"Bach, Johann Sebastian","Invention No. 10 in G major, BWV 781",None,NaN,D3


## 3.3 Visualizing Results for Corpus

### 3.3.1 A Bar Chart of Notes for a Corpus

Here we look at the distribution of notes, breaking things down by voice-part (1 is the top part, 2 is the bottom one)

In [32]:

# run function and create chart
corpus_note_df = corpus.notes(combine_unisons_choice=True, combine_rests_choice=False)

# now we need to 'melt' the data around the voice parts to make things easier!
value_vars= ['1', '2']
id_vars = [col for col in corpus_note_df.columns if col not in value_vars]
corpus_note_df_melted = pd.melt(corpus_note_df, value_vars=value_vars, id_vars=id_vars)
corpus_note_df_melted.rename(columns={'variable': 'Voice', 'value': 'Note'}, inplace=True)

# Clean data - remove Rest and NaN values
df_clean = corpus_note_df_melted[
    (corpus_note_df_melted['Note'].notna()) & 
    (corpus_note_df_melted['Note'] != 'Rest')
].copy()

# Count notes by Composer Title Note and Voice
corpus_note_counts = df_clean.groupby(['Composer', 'Title', 'Note', 'Voice']).size().reset_index(name='Count')
corpus_note_counts.head(5)


,Composer,Title,Note,Voice,Count
0,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",A2,2,3
1,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",A3,2,21
2,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",A4,1,22
3,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",A4,2,5
4,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",A5,1,10


In [33]:
# Create the bar chart

# sort the notes that actually occur in this piece, low to high
note_order = sort_pitch_values(corpus_note_counts['Note'].unique().tolist())

fig = px.bar(
    corpus_note_counts,
    x='Note',
    y='Count',
    color='Title',
    height=800,
    width=1000,
    barmode='stack',
    # note that the pitch order is set earlier in our NB!
    category_orders={'Note': note_order}
)

fig.show()

### 3.3.2 As Radar Plot

Here we display the notes on a radar plot.  But now the colors correspond to the title of each piece.

Of course we could set the color to correspond to some other feature in the df, like composer.  

In [34]:

# # Option to remove rests
corpus_clean = corpus_note_counts[corpus_note_counts['Note'] != 'Rest']

# remove octave designation
corpus_clean['Note'] = corpus_clean['Note'].str.replace(r'\d+', '', regex=True)

# # Function to standardize note names
def standardize_note(note):
   
    if '-' in note:
        return note.replace('-', 'b')
    return note

corpus_clean['Note'] = corpus_clean['Note'].apply(standardize_note)

# Count notes by Composer Title Note (but NOT voice)
corpus_note_counts_grouped = corpus_clean.groupby(['Composer', 'Title', 'Note', ]).size().reset_index(name='Count')


# now let's scale the counts relative to the total number of notes in the piece
total_notes = corpus_note_counts_grouped['Count'].sum()
corpus_note_counts_grouped['Scaled_Count'] = corpus_note_counts_grouped['Count']/total_notes
corpus_note_counts_grouped.head(5)

,Composer,Title,Note,Count,Scaled_Count
0,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",A,5,0.040
1,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",B,4,0.032
2,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",Bb,3,0.024
3,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",C,6,0.048
4,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",C#,2,0.016


In [35]:
# categorical sorting:  'fifths' or 'chromatic'
pitch_class_order = fifths

corpus_note_counts_grouped['Note'] = pd.Categorical(corpus_note_counts_grouped['Note'], categories=pitch_class_order, ordered=True)
corpus_note_counts_grouped_sorted = corpus_note_counts_grouped.sort_values('Note').reset_index(drop=True)
corpus_note_counts_grouped_sorted

,Composer,Title,Note,Count,Scaled_Count
0,"Bach, Johann Sebastian","Invention No. 3 in D major, BWV 774",C,1,0.008
1,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",C,6,0.048
2,"Bach, Johann Sebastian","Invention No. 10 in G major, BWV 781",C,4,0.032
3,"Bach, Johann Sebastian","Invention No. 10 in G major, BWV 781",G,6,0.048
4,"Bach, Johann Sebastian","Invention No. 3 in D major, BWV 774",G,5,0.040
5,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",G,5,0.040
6,"Bach, Johann Sebastian","Invention No. 10 in G major, BWV 781",D,5,0.040
7,"Bach, Johann Sebastian","Invention No. 3 in D major, BWV 774",D,5,0.040
8,"Bach, Johann Sebastian","Invention No. 1 in C major, BWV 772",D,5,0.040
9,"Bach, Johann Sebastian","Invention No. 3 in D major, BWV 774",A,6,0.048


In [36]:
titles = corpus_note_counts_grouped_sorted['Title'].unique()

# drop anything not in the pitch list (e.g. 'Rest'), which would otherwise show up as a blank spoke
corpus_plot_df = corpus_note_counts_grouped_sorted[corpus_note_counts_grouped_sorted['Note'].isin(fifths)]

# keep only the pitches that occur somewhere in the corpus, in the order given by fifths
notes_present = [n for n in fifths if n in corpus_plot_df['Note'].values]

fig = px.line_polar(
    corpus_plot_df,
    r='Scaled_Count',
    theta='Note',
    color='Title',
    line_close=True,
    range_r=[0, corpus_plot_df['Scaled_Count'].max() * 1.1],
    markers=True,
    height=800,
    width=800,
    category_orders={'Note': notes_present}
)
fig.update_traces(fill='toself', 
                mode='markers+lines',
                opacity=.7)
fig.update_layout(
    showlegend=True,
    legend=dict(
        title='Title',
        orientation="v"
    ),
    title = 'Weighted Note Distribution in Corpus')
fig.show()


### 3.3.3. Barchart of Melodic Intervals

In [37]:
mel = corpus.mel(kind_choice='d', compound_choice=False)
# Melt the dataframe to combine columns '1' and '2' into one
mel_melted = mel.melt(
    id_vars=['Composer', 'Title', 'Date'], 
    value_vars=['1', '2'], 
    var_name='Voice', 
    value_name='Interval'
)

# Drop NaN values
mel_melted = mel_melted.dropna(subset=['Interval'])

# Count intervals by Title (and optionally Voice)
mel_counts = mel_melted.groupby(['Title', 'Interval']).size().reset_index(name='Count')

# Get unique intervals and convert to integers for sorting
unique_intervals = mel_counts['Interval'].dropna().unique()

# Sort: negatives descending (most negative first), then positives ascending
def interval_sort_key(x):
    try:
        val = int(x)
        # Negatives get sorted by their value (ascending, so -8 comes before -7)
        # Positives get sorted after all negatives
        if val < 0:
            return (0, val)  # First group, sort by value
        else:
            return (1, val)  # Second group, sort by value
    except:
        return (2, 0)  # Put non-numeric (like 'Rest') at the end

sorted_intervals = sorted(unique_intervals, key=interval_sort_key)

In [38]:

chart_title = "Distribution of Melodic Intervals in Corpus"

#create barchart
fig = px.bar(
    mel_counts,
    x='Interval',
    y='Count',
    color='Title',
    category_orders={'Interval': sorted_intervals}
)
fig.update_layout(xaxis_title="Interval", 
                    yaxis_title="Count",
                    legend_title='Voice',
                       height=500,
                       width=750)

fig.show()

## 4.0 Network of Pieces based on Shared Melodic Ngrams


Here we build a **network** of pieces:  each time two pieces share a melodic ngram, they get an edge.

- The results are often quite dense, so we include a 'minimum threshold' value:  unless two pieces have at least 'n' edges, they are excluded from the network.

- A 'thickness' adjustment also scales the width of the edges so that the results are easier to interpret.

```python
# the ngram conditions
n=4
combineUnisons=False
kind='c'
thematic = True
anywhere = False
threshold_for_shared_ngrams = 30
thickness_adjust = 5
```

In [39]:
# Network Function--Do Not Edit!
def ng_network(data, 
               threshold_for_shared_ngrams=50, 
               thickness_adjust=10, 
               network_name='my_network.html'):

    # and now, a network in which the nodes are the pieces and edges represent the ngrams they share.  
    # the thickness of the edges varies with the number of shared ngrams
    # the colors distinguish 'communities' of pieces that are highly related

    df = pd.DataFrame(data)
    df = df.reset_index()
    df.drop('level_1', axis=1, inplace=True)
    df = df.rename(columns={0: 'ngram'})

    #define the function to convert tuples to strings
    def convertTuple(tup):
        out = ""
        if isinstance(tup, tuple):
            out = '_'.join(tup)
        return out  
    # clean the tuples
    df['ngram'] = df['ngram'].apply(convertTuple)

    # Group by 'ngram' and extract a list of unique titles for each group
    grouped_titles = df.groupby('ngram')['title'].unique().reset_index(name='titles')

    # Generate all pairs of titles for each group
    all_pairs = []
    for _, row in grouped_titles.iterrows():
        pairs = list(combinations(row['titles'], 2))
        all_pairs.append((row['ngram'], pairs))

    # Create a new DataFrame with the results
    result_df = pd.DataFrame(all_pairs, columns=['ngram', 'title_pairs'])
    # remove the empty pairs
    df_filtered = result_df[result_df['title_pairs'].apply(len) > 0]

    # explode the complicated lists of tuples, effectively 'tyding' the data
    exploded_df = df_filtered.explode('title_pairs')

    # get the counts of each pair, which provides the basis of the weights
    pair_counts = exploded_df['title_pairs'].value_counts()

    # limit to high scoring pairs (>3)
    pair_counts = pair_counts[pair_counts >= threshold_for_shared_ngrams]

    # Adding Louvain Communities
    def add_communities(G):
        G = deepcopy(G)
        partition = community_louvain.best_partition(G)
        nx.set_node_attributes(G, partition, "group")
        return G

    # Create an empty NetworkX graph
    G = nx.Graph()


    # Add nodes and assign weights to edges
    for pair, count in pair_counts.items():
        # Directly unpacking the tuple into node1 and node2
        node1, node2 = pair
        # Adding nodes if they don't exist already
        if node1 not in G.nodes:
            G.add_node(node1)
        if node2 not in G.nodes:
            G.add_node(node2)
        # Adding edge with weight
        G.add_edge(node1, node2, weight=count)

    # Adjusting edge thickness based on weights
    for edge in G.edges(data=True):
        edge[2]['width'] = edge[2]['weight']/thickness_adjust

    G = add_communities(G)

    # set display parameters
    ngram_map = Network(notebook=True,
                       width="800",
                              height="800",
                              bgcolor="black", 
                              font_color="white",
                              cdn_resources="in_line")

    # Set the physics layout of the network
    ngram_map.set_options("""
    {
    "physics": {
    "enabled": true,
    "forceAtlas2Based": {
        "springLength": 1
    },
    "solver": "forceAtlas2Based"
    }
    }
    """)

    ngram_map.from_nx(G)
    return ngram_map.show(network_name)
    

    

### Make the Network

NOTE:  you will need to find the HTML file in your folder and open it in a new browser tab!

In [36]:


# the ngram conditions
n=6
combineUnisons=False
kind='c'
thematic = True
anywhere = False
threshold_for_shared_ngrams = 30
thickness_adjust = 25
network_name = 'melodic_ngram_network.html'


# now we build a list of all of the ngrams in each piece
list_series = []
for item in all_bach:
    piece = importScore(item)
    title = piece.metadata['composer'] + ": " + piece.metadata['title']
    # find entries for model
    nr = piece.notes(combineUnisons=combineUnisons)
    nr = piece.numberParts(df=nr)
    mel = piece.melodic(df=nr, kind=kind, compound=False, unit=0, end=False)
    mel_ngrams = piece.ngrams(df=mel, n=n, exclude=['Rest'])
    mel_ngrams.fillna('')
    mel_ngrams['title'] = title
    mel_ngrams.set_index(['title'], inplace=True)
    # Set the new column as the index
    mel_stacked_ngs = mel_ngrams.stack()
    list_series.append(mel_stacked_ngs)

ngs = pd.concat(list_series)

print("Network of Melodic Ngrams")
ng_network(ngs, threshold_for_shared_ngrams, thickness_adjust, network_name)


Network of Melodic Ngrams
melodic_ngram_network.html
